
# Trustworthy Cardiovascular Prediction with Quantitative Explainability Evaluation
## Final GitHub Master Codebook

This notebook is the **final frozen research codebook** for the study.

It consolidates the validated workflow for:

1. **Dataset A — binary CVD prediction**
   - physiological quality control
   - 80/20 stratified development/locked-test split
   - nine baseline models
   - calibrated HGB and EBM branches
   - frozen HGB–EBM hybrid:  
     \[
     P(CVD)=0.75P_{HGB}+0.25P_{EBM}
     \]
   - locked-test evaluation

2. **Dataset A — quantitative explainability**
   - HGB-SHAP
   - intrinsic EBM explanations
   - global and patient-level agreement
   - explanation-space stability
   - corrected feature-space fidelity
   - feature-space stability
   - age-stratified robustness

3. **Dataset B — ordered risk stratification**
   - leakage/redundancy control
   - Nominal Random Forest
   - Ordinal XGBoost
   - EBM comparator
   - strict 5-fold OOF prediction
   - **cross-fitted isotonic calibration**
   - bootstrap confidence intervals
   - HIGH-risk SHAP analysis

4. **Cross-dataset explanation concordance**
   - nine harmonized concepts
   - Spearman and Kendall rank agreement
   - Top-3 / Top-5 overlap
   - mean absolute rank difference
   - exact/permutation inference

### Important interpretation boundaries

- Outputs are **model-based CVD predictions or risk-stratification estimates**, not diagnoses.
- Dataset B is **not** an external validation dataset for Dataset A.
- SHAP/EBM attributions describe **learned model behaviour**, not causal effects.
- The study does not claim clinical deployment readiness.
- The final Model-B results are the **strict cross-fitted results**, not earlier same-OOF calibration results.
- The corrected Phase-3C fidelity experiment uses the **same random comparator sets for point estimates and bootstrap intervals**.

### Authoritative source phases incorporated

- `CVD_Q1_Final_Master_Codebook (1).ipynb`
- `CVD_ModelB_Phase3B_CrossFitted_Calibration_EBM_Validation (1).ipynb`
- `CVD_Phase3C_Corrected_Fidelity_All_In_One.ipynb`
- `CVD_Reviewer_Defence_Complete_Inference.ipynb`

The final frozen values at the end of this notebook should be treated as the manuscript values.



## 0. Reproducibility and repository setup

Recommended repository structure:

```text
repository/
├── CVD_Q1_Final_GitHub_Master_Notebook.ipynb
├── data/
│   ├── cardio_train.csv
│   └── CVD Dataset.csv
├── outputs/
└── README.md
```

The raw datasets may be excluded from Git if their licenses or size constraints require it.


In [ ]:

# Optional installation cell for a clean Colab/kernel.
# Uncomment when required.

# %pip install -q numpy pandas scipy scikit-learn matplotlib lightgbm xgboost shap interpret


In [ ]:

from pathlib import Path
import os, json, zipfile, warnings, math, itertools
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, cross_val_predict
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
)
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    balanced_accuracy_score, recall_score, precision_score,
    f1_score, matthews_corrcoef, brier_score_loss, log_loss,
    confusion_matrix
)
from scipy.stats import spearmanr, kendalltau, hypergeom

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from interpret.glassbox import ExplainableBoostingClassifier
import shap

SEED = 42
EPS = 1e-6
np.random.seed(SEED)

ROOT = Path(".")
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

MODELA_CSV = DATA_DIR / "cardio_train.csv"
MODELB_CSV = DATA_DIR / "CVD Dataset.csv"

print("Environment ready.")
print("Output directory:", OUTPUT_DIR.resolve())


# PART I — DATASET A: BINARY CVD PREDICTION

## 1. Load Dataset A

In [ ]:

def load_model_a(path=MODELA_CSV):
    path = Path(path)
    if path.exists():
        df = pd.read_csv(path, sep=";")
        if df.shape[1] == 1:
            df = pd.read_csv(path)
        return df

    # Accept common repository/runtime alternatives.
    candidates = [
        Path("cardio_train.csv"),
        Path("/content/cardio_train.csv"),
        Path("data/cardio_train.csv"),
    ]
    for p in candidates:
        if p.exists():
            df = pd.read_csv(p, sep=";")
            if df.shape[1] == 1:
                df = pd.read_csv(p)
            return df

    raise FileNotFoundError(
        "Dataset A not found. Place cardio_train.csv under ./data/."
    )

cardio = load_model_a()
print("Dataset A shape:", cardio.shape)
display(cardio.head())


## 2. Dataset-A quality-control audit

In [ ]:
qc_A = {
    "N": len(cardio),
    "No_CVD": int((cardio["cardio"] == 0).sum()),
    "CVD": int((cardio["cardio"] == 1).sum()),
    "Invalid_Height": int((~cardio["height"].between(120, 220)).sum()),
    "Invalid_Weight": int((~cardio["weight"].between(30, 200)).sum()),
    "Invalid_SBP": int((~cardio["ap_hi"].between(70, 250)).sum()),
    "Invalid_DBP": int((~cardio["ap_lo"].between(40, 150)).sum()),
    "SBP_LE_DBP": int((cardio["ap_hi"] <= cardio["ap_lo"]).sum())
}

display(pd.Series(qc_A, name="Value"))


## 3. Leakage-safe cleaning, feature engineering, and locked split

In [ ]:
A = cardio.copy()

A["age_years"] = A["age"] / 365.25

clinical_ranges = {
    "height": (120, 220),
    "weight": (30, 200),
    "ap_hi": (70, 250),
    "ap_lo": (40, 150)
}

for col, (low, high) in clinical_ranges.items():
    A.loc[~A[col].between(low, high), col] = np.nan

bad_bp = (
    A["ap_hi"].notna()
    & A["ap_lo"].notna()
    & (A["ap_hi"] <= A["ap_lo"])
)

A.loc[bad_bp, ["ap_hi", "ap_lo"]] = np.nan

A["bmi"] = A["weight"] / (A["height"] / 100) ** 2
A.loc[~A["bmi"].between(10, 80), "bmi"] = np.nan

A["pulse_pressure"] = A["ap_hi"] - A["ap_lo"]
A["map_est"] = A["ap_lo"] + (A["ap_hi"] - A["ap_lo"]) / 3

A_num = [
    "age_years",
    "height",
    "weight",
    "ap_hi",
    "ap_lo",
    "bmi",
    "pulse_pressure",
    "map_est"
]

A_cat = [
    "gender",
    "cholesterol",
    "gluc",
    "smoke",
    "alco",
    "active"
]

A_features = A_num + A_cat

X_A = A[A_features].copy()
y_A = A["cardio"].astype(int)

X_A_train, X_A_test, y_A_train, y_A_test = train_test_split(
    X_A,
    y_A,
    test_size=0.20,
    stratify=y_A,
    random_state=SEED
)

print("Development:", X_A_train.shape)
print("Locked test:", X_A_test.shape)


## 4. Common preprocessing

In [ ]:
A_num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

A_cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

A_pre = ColumnTransformer(
    transformers=[
        ("numeric", A_num_pipe, A_num),
        ("categorical", A_cat_pipe, A_cat)
    ],
    sparse_threshold=0
)


## 5. Nine-model baseline benchmark

In [ ]:
A_baselines = {
    "Logistic Regression":
        LogisticRegression(max_iter=3000, random_state=SEED),

    "Gaussian Naive Bayes":
        GaussianNB(),

    "KNN":
        KNeighborsClassifier(n_neighbors=25, weights="distance"),

    "Decision Tree":
        DecisionTreeClassifier(
            max_depth=6,
            min_samples_leaf=20,
            class_weight="balanced",
            random_state=SEED
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=5,
            class_weight="balanced",
            n_jobs=-1,
            random_state=SEED
        ),

    "Extra Trees":
        ExtraTreesClassifier(
            n_estimators=300,
            max_depth=14,
            min_samples_leaf=5,
            class_weight="balanced",
            n_jobs=-1,
            random_state=SEED
        ),

    "HistGradientBoosting":
        HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=250,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=SEED
        ),

    "LightGBM":
        LGBMClassifier(
            n_estimators=250,
            learning_rate=0.04,
            num_leaves=31,
            max_depth=8,
            min_child_samples=40,
            reg_lambda=1.0,
            verbosity=-1,
            n_jobs=-1,
            random_state=SEED
        ),

    "XGBoost":
        XGBClassifier(
            n_estimators=300,
            learning_rate=0.04,
            max_depth=5,
            min_child_weight=5,
            subsample=0.90,
            colsample_bytree=0.90,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            n_jobs=-1,
            random_state=SEED
        )
}

A_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

A_baseline_rows = []

for name, clf in A_baselines.items():

    pipe = Pipeline([
        ("preprocessor", A_pre),
        ("classifier", clf)
    ])

    scores = cross_validate(
        pipe,
        X_A_train,
        y_A_train,
        cv=A_cv,
        scoring={
            "AUROC": "roc_auc",
            "AUPRC": "average_precision",
            "F1": "f1",
            "Balanced_Accuracy": "balanced_accuracy"
        },
        n_jobs=1
    )

    A_baseline_rows.append({
        "Model": name,
        "CV_AUROC_mean": scores["test_AUROC"].mean(),
        "CV_AUROC_sd": scores["test_AUROC"].std(ddof=1),
        "CV_AUPRC_mean": scores["test_AUPRC"].mean(),
        "CV_F1_mean": scores["test_F1"].mean(),
        "CV_Balanced_Accuracy_mean":
            scores["test_Balanced_Accuracy"].mean()
    })

A_baseline_table = pd.DataFrame(A_baseline_rows).sort_values(
    "CV_AUROC_mean",
    ascending=False
)

display(A_baseline_table)

A_baseline_table.to_csv(
    OUTPUT_DIR / "MASTER_Table_ModelA_Baselines.csv",
    index=False
)


## 6. Development OOF probabilities for HGB and EBM

In [ ]:
A_hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=250,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=SEED
)

A_ebm = ExplainableBoostingClassifier(
    interactions=10,
    max_bins=256,
    max_rounds=3000,
    learning_rate=0.03,
    outer_bags=8,
    inner_bags=0,
    random_state=SEED,
    n_jobs=-1
)

A_hgb_pipe = Pipeline([
    ("preprocessor", A_pre),
    ("classifier", A_hgb)
])

A_ebm_pipe = Pipeline([
    ("preprocessor", A_pre),
    ("classifier", A_ebm)
])

print("Generating HGB OOF...")
A_hgb_oof = cross_val_predict(
    A_hgb_pipe,
    X_A_train,
    y_A_train,
    cv=A_cv,
    method="predict_proba",
    n_jobs=1
)[:,1]

print("Generating EBM OOF...")
A_ebm_oof = cross_val_predict(
    A_ebm_pipe,
    X_A_train,
    y_A_train,
    cv=A_cv,
    method="predict_proba",
    n_jobs=1
)[:,1]


## 7. Frozen calibration mappings

In [ ]:
def clip_prob(p):
    return np.clip(np.asarray(p, dtype=float), EPS, 1-EPS)

def logit(p):
    p = clip_prob(p)
    return np.log(p / (1-p))

A_hgb_platt = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=3000
)

A_hgb_platt.fit(
    logit(A_hgb_oof).reshape(-1,1),
    y_A_train.to_numpy()
)

A_ebm_iso = IsotonicRegression(
    out_of_bounds="clip",
    y_min=EPS,
    y_max=1-EPS
)

A_ebm_iso.fit(
    A_ebm_oof,
    y_A_train.to_numpy()
)

print("Calibration mappings fitted.")


## 8. Final locked-test evaluation of HGB, EBM, and hybrid

In [ ]:
A_hgb_pipe.fit(X_A_train, y_A_train)
A_ebm_pipe.fit(X_A_train, y_A_train)

pA_hgb_raw = clip_prob(
    A_hgb_pipe.predict_proba(X_A_test)[:,1]
)

pA_ebm_raw = clip_prob(
    A_ebm_pipe.predict_proba(X_A_test)[:,1]
)

pA_hgb = A_hgb_platt.predict_proba(
    logit(pA_hgb_raw).reshape(-1,1)
)[:,1]

pA_ebm = A_ebm_iso.predict(pA_ebm_raw)

pA_h2 = (
    0.75 * pA_hgb
    + 0.25 * pA_ebm
)

def binary_metrics(y_true, p, threshold=0.50):
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        pred,
        labels=[0,1]
    ).ravel()

    return {
        "AUROC": roc_auc_score(y_true, p),
        "AUPRC": average_precision_score(y_true, p),
        "Brier": brier_score_loss(y_true, p),
        "LogLoss": log_loss(y_true, p),
        "Accuracy": accuracy_score(y_true, pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, pred),
        "Sensitivity": recall_score(y_true, pred),
        "Specificity": tn / (tn + fp),
        "Precision": precision_score(y_true, pred),
        "F1": f1_score(y_true, pred),
        "MCC": matthews_corrcoef(y_true, pred)
    }

A_final_table = pd.DataFrame([
    {
        "Model": "HGB calibrated",
        **binary_metrics(y_A_test, pA_hgb)
    },
    {
        "Model": "EBM calibrated",
        **binary_metrics(y_A_test, pA_ebm)
    },
    {
        "Model": "H2 frozen hybrid",
        **binary_metrics(y_A_test, pA_h2)
    }
])

display(A_final_table)

A_final_table.to_csv(
    OUTPUT_DIR / "MASTER_Table_ModelA_Final.csv",
    index=False
)



### Frozen Model-A locked-test values used in the manuscript

These are retained as an audit target after rerunning the pipeline.


In [ ]:

MODELA_FROZEN = pd.DataFrame([
    {
        "Model": "Calibrated HGB",
        "AUROC": 0.799743,
        "AUPRC": 0.783543,
        "Brier": 0.181112,
        "LogLoss": 0.542529,
        "Accuracy": 0.733786,
        "Balanced_Accuracy": 0.733764,
        "Sensitivity": 0.695683,
        "Specificity": 0.771845,
        "Precision": 0.752823,
        "F1": 0.723126,
        "MCC": 0.468899,
    },
    {
        "Model": "Calibrated EBM",
        "AUROC": 0.797102,
        "AUPRC": 0.774257,
        "Brier": 0.182375,
        "LogLoss": 0.545502,
        "Accuracy": 0.730929,
        "Balanced_Accuracy": 0.730897,
        "Sensitivity": 0.675529,
        "Specificity": 0.786265,
        "Precision": 0.759441,
        "F1": 0.715031,
        "MCC": 0.464665,
    },
    {
        "Model": "Proposed HGB-EBM Hybrid",
        "AUROC": 0.799982,
        "AUPRC": 0.784394,
        "Brier": 0.181069,
        "LogLoss": 0.542384,
        "Accuracy": 0.732571,
        "Balanced_Accuracy": 0.732551,
        "Sensitivity": 0.695969,
        "Specificity": 0.769132,
        "Precision": 0.750694,
        "F1": 0.722296,
        "MCC": 0.466360,
    },
])

display(MODELA_FROZEN)
MODELA_FROZEN.to_csv(OUTPUT_DIR / "FINAL_Table_I_ModelA_LockedTest.csv", index=False)


# PART II — DATASET A: EXPLAINABILITY

## 9. HGB-SHAP global ranking on the locked test set

In [ ]:
# Extract transformed HGB model
A_pre_fitted = A_hgb_pipe.named_steps["preprocessor"]
A_hgb_fitted = A_hgb_pipe.named_steps["classifier"]

XA_test_t = A_pre_fitted.transform(X_A_test)
A_feature_names_t = list(
    A_pre_fitted.get_feature_names_out()
)

A_explainer = shap.TreeExplainer(
    A_hgb_fitted
)

A_shap_raw = A_explainer.shap_values(
    XA_test_t
)

if isinstance(A_shap_raw, list):
    A_shap = np.asarray(A_shap_raw[-1])
else:
    A_shap = np.asarray(A_shap_raw)

if A_shap.ndim == 3:
    A_shap = A_shap[:,:, -1]

def A_transformed_to_concept(name):
    if name.startswith("numeric__"):
        return name.replace("numeric__", "")

    if name.startswith("categorical__"):
        stripped = name.replace("categorical__", "")

        for parent in A_cat:
            if stripped == parent or stripped.startswith(parent + "_"):
                return parent

    return name

A_concepts_t = [
    A_transformed_to_concept(n)
    for n in A_feature_names_t
]

A_concept_shap = pd.DataFrame(
    0.0,
    index=np.arange(len(X_A_test)),
    columns=A_features
)

for j, concept in enumerate(A_concepts_t):
    if concept in A_concept_shap.columns:
        A_concept_shap[concept] += A_shap[:,j]

A_global_rank = (
    A_concept_shap
    .abs()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

A_global_rank.columns = [
    "Clinical_Concept",
    "Mean_Abs_SHAP"
]

A_global_rank["Rank"] = (
    np.arange(len(A_global_rank)) + 1
)

display(A_global_rank)

A_global_rank.to_csv(
    OUTPUT_DIR / "MASTER_ModelA_Global_SHAP_Ranking.csv",
    index=False
)



## 10. Final quantitative XAI agreement and explanation-space stability

The full patient-level inference workflow was finalized in the reviewer-defence phase.
The table below is the frozen Table-II output used in the paper.


In [ ]:

TABLE_II_FINAL = pd.DataFrame([
    ["Global Spearman rho", 0.851, 0.851, 0.890, "<0.001"],
    ["Global Kendall tau", 0.736, 0.736, 0.802, "<0.001"],
    ["Global Top-3 overlap", 1.000, 1.000, 1.000, "0.003"],
    ["Global Top-5 overlap", 0.800, 0.800, 0.800, "0.023"],
    ["Patient-level Spearman", 0.744, 0.742, 0.747, "<0.001"],
    ["Patient-level cosine similarity", 0.946, 0.945, 0.947, "<0.001"],
    ["Patient-level Top-3 overlap", 0.734, 0.731, 0.738, "<0.001"],
    ["Patient-level Top-5 overlap", 0.787, 0.785, 0.790, "<0.001"],
    ["Attribution-sign agreement", 0.852, 0.851, 0.854, "<0.001"],
    ["HGB explanation-space Top-5 stability", 0.987, 0.987, 0.988, "<0.001"],
    ["EBM explanation-space Top-5 stability", 0.991, 0.991, 0.992, "<0.001"],
], columns=["Measure", "Estimate", "CI_Lower_95", "CI_Upper_95", "p"])

display(TABLE_II_FINAL)
TABLE_II_FINAL.to_csv(OUTPUT_DIR / "FINAL_Table_II_XAI_Agreement.csv", index=False)



## 11. Corrected feature-space fidelity — final authoritative results

The earlier Phase-3C implementation used inconsistent random-comparator seeds between point estimates and bootstrap intervals.  
The corrected experiment uses the **same paired random comparator sets** for both.

Authoritative notebook: `CVD_Phase3C_Corrected_Fidelity_All_In_One.ipynb`.


In [ ]:

FIDELITY_FINAL = pd.DataFrame([
    ["HGB-SHAP", 1, -0.004540, -0.006436, -0.002739],
    ["HGB-SHAP", 3,  0.032664,  0.029440,  0.036013],
    ["HGB-SHAP", 5,  0.033054,  0.029550,  0.036600],
    ["EBM",      1,  0.002483,  0.000843,  0.004096],
    ["EBM",      3,  0.038034,  0.035103,  0.040859],
    ["EBM",      5,  0.053533,  0.050393,  0.056599],
], columns=["Explainer", "Top_k", "Mean_Fidelity_Gain", "CI_Lower_95", "CI_Upper_95"])

FEATURE_SPACE_STABILITY = pd.DataFrame([
    ["HGB-SHAP", 0.976667],
    ["EBM", 0.963333],
], columns=["Explainer", "Top5_Jaccard_Stability"])

display(FIDELITY_FINAL)
display(FEATURE_SPACE_STABILITY)

FIDELITY_FINAL.to_csv(OUTPUT_DIR / "FINAL_Phase3C_Corrected_Fidelity.csv", index=False)
FEATURE_SPACE_STABILITY.to_csv(OUTPUT_DIR / "FINAL_FeatureSpace_Stability.csv", index=False)


## 12. Final age-stratified robustness

In [ ]:

AGE_ROBUSTNESS_FINAL = pd.DataFrame([
    ["<40",   0.819494, 0.809473, 0.828079, 0.5248, 0.9639],
    ["40-49", 0.821477, 0.771784, 0.807398, 0.5910, 0.9068],
    ["50-59", 0.770588, 0.741563, 0.779195, 0.6574, 0.7659],
    [">=60",  0.711640, 0.699529, 0.772619, 0.8804, 0.3260],
], columns=[
    "Age_Group", "AUROC", "Explanation_Spearman",
    "Top5_Overlap", "Sensitivity", "Specificity"
])

display(AGE_ROBUSTNESS_FINAL)
AGE_ROBUSTNESS_FINAL.to_csv(OUTPUT_DIR / "FINAL_Age_Subgroup_Robustness.csv", index=False)

print(">=60 subgroup prevalence = 0.668")
print(">=60 predicted-positive proportion ≈ 0.674")
print(">=60 calibration intercept = -0.0088")
print(">=60 calibration slope = 0.9613")


# PART III — DATASET B: STRICT CROSS-FITTED ORDERED RISK STRATIFICATION


The following section is taken from the **authoritative corrective codebook**:

`CVD_ModelB_Phase3B_CrossFitted_Calibration_EBM_Validation (1).ipynb`

This is the source of the final Table-III values.


## 13. Load Dataset B

In [ ]:

def load_model_b(path=MODELB_CSV):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)

    candidates = [
        Path("CVD Dataset.csv"),
        Path("/content/CVD Dataset.csv"),
        Path("data/CVD Dataset.csv"),
    ]
    for p in candidates:
        if p.exists():
            return pd.read_csv(p)

    raise FileNotFoundError(
        "Dataset B not found. Place 'CVD Dataset.csv' under ./data/."
    )

df = load_model_b()
print("Dataset B shape:", df.shape)
print(df["CVD Risk Level"].value_counts())


## 14. Frozen primary feature set and leakage-aware inputs

In [ ]:
aliases = {
    "Age": ["Age"],
    "Sex": ["Sex"],
    "Weight": ["Weight", "Weight (kg)"],
    "Height_cm": ["Height (cm)", "Height"],
    "Abdominal Circumference": [
        "Abdominal Circumference",
        "Abdominal Circumference (cm)"
    ],
    "Total Cholesterol": [
        "Total Cholesterol",
        "Total Cholesterol (mg/dL)"
    ],
    "HDL": ["HDL", "HDL (mg/dL)"],
    "Fasting Blood Sugar": [
        "Fasting Blood Sugar",
        "Fasting Blood Sugar (mg/dL)"
    ],
    "Systolic BP": ["Systolic BP"],
    "Diastolic BP": ["Diastolic BP"],
    "Smoking Status": ["Smoking Status"],
    "Diabetes Status": ["Diabetes Status"],
    "Physical Activity Level": ["Physical Activity Level"],
    "Family History of CVD": ["Family History of CVD"]
}

resolved = {}

for concept, candidates in aliases.items():
    found = [c for c in candidates if c in df.columns]

    if found:
        resolved[concept] = found[0]

primary_concepts = list(aliases.keys())

missing = [
    c for c in primary_concepts
    if c not in resolved
]

if missing:
    raise ValueError(
        f"Missing required concepts: {missing}"
    )

source_cols = [
    resolved[c]
    for c in primary_concepts
]

X = df[source_cols].copy()

y_raw = (
    df["CVD Risk Level"]
    .astype(str)
    .str.strip()
)

y = y_raw.map(
    CLASS_TO_INT
).astype(int)

print("Features:", primary_concepts)
print(y.value_counts().sort_index())


## 15. Dataset-B preprocessing

In [ ]:
categorical_concepts = {
    "Sex",
    "Smoking Status",
    "Diabetes Status",
    "Physical Activity Level",
    "Family History of CVD"
}

numeric_features = [
    resolved[c]
    for c in primary_concepts
    if c not in categorical_concepts
]

categorical_features = [
    resolved[c]
    for c in primary_concepts
    if c in categorical_concepts
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    sparse_threshold=0
)

print("Preprocessing configured.")


## 16. Five-fold OOF configuration

In [ ]:
oof_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)


### 16.1 Nominal Random Forest OOF probabilities

In [ ]:
# Nominal RF OOF

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1
)

rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", rf)
])

print("Generating RF OOF probabilities...")

rf_oof = cross_val_predict(
    rf_pipe,
    X,
    y,
    cv=oof_cv,
    method="predict_proba",
    n_jobs=1
)

print(rf_oof.shape)


### 16.2 Ordinal XGBoost OOF probabilities

In [ ]:
def xgb_binary_factory():
    return XGBClassifier(
        n_estimators=300,
        learning_rate=0.04,
        max_depth=5,
        min_child_weight=3,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=SEED,
        n_jobs=-1
    )

def ordinal_oof_predict(
    X_df,
    y_multiclass,
    preprocessor,
    cv
):
    n = len(y_multiclass)

    p_gt_low = np.zeros(
        n,
        dtype=float
    )

    p_gt_mid = np.zeros(
        n,
        dtype=float
    )

    for train_idx, val_idx in cv.split(
        X_df,
        y_multiclass
    ):
        X_tr = X_df.iloc[
            train_idx
        ]

        X_va = X_df.iloc[
            val_idx
        ]

        y_tr = y_multiclass.iloc[
            train_idx
        ]

        t1 = (
            y_tr > 0
        ).astype(int)

        t2 = (
            y_tr > 1
        ).astype(int)

        m1 = Pipeline([
            (
                "preprocessor",
                preprocessor
            ),
            (
                "classifier",
                xgb_binary_factory()
            )
        ])

        m2 = Pipeline([
            (
                "preprocessor",
                preprocessor
            ),
            (
                "classifier",
                xgb_binary_factory()
            )
        ])

        m1.fit(
            X_tr,
            t1
        )

        m2.fit(
            X_tr,
            t2
        )

        p1 = m1.predict_proba(
            X_va
        )[:,1]

        p2 = m2.predict_proba(
            X_va
        )[:,1]

        p2 = np.minimum(
            p2,
            p1
        )

        p_gt_low[
            val_idx
        ] = p1

        p_gt_mid[
            val_idx
        ] = p2

    p_low = (
        1 - p_gt_low
    )

    p_high = (
        p_gt_mid
    )

    p_mid = (
        p_gt_low
        - p_gt_mid
    )

    prob = np.column_stack([
        p_low,
        p_mid,
        p_high
    ])

    prob = np.clip(
        prob,
        0,
        1
    )

    row_sum = prob.sum(
        axis=1,
        keepdims=True
    )

    row_sum[
        row_sum == 0
    ] = 1

    return (
        prob
        / row_sum
    )

print("Generating ordinal XGBoost OOF probabilities...")

ordxgb_oof = ordinal_oof_predict(
    X,
    y,
    preprocessor,
    oof_cv
)

print(ordxgb_oof.shape)


### 16.3 EBM OOF probabilities

In [ ]:
# Multiclass EBM OOF

ebm = ExplainableBoostingClassifier(
    interactions=10,
    max_bins=256,
    max_rounds=3000,
    learning_rate=0.03,
    outer_bags=8,
    inner_bags=0,
    random_state=SEED,
    n_jobs=-1
)

ebm_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", ebm)
])

print("Generating EBM OOF probabilities...")

ebm_oof = cross_val_predict(
    ebm_pipe,
    X,
    y,
    cv=oof_cv,
    method="predict_proba",
    n_jobs=1
)

print(ebm_oof.shape)


## 17. Probability normalization helpers

In [ ]:
def renormalize(prob):
    prob = np.clip(
        np.asarray(
            prob,
            dtype=float
        ),
        EPS,
        1-EPS
    )

    s = prob.sum(
        axis=1,
        keepdims=True
    )

    s[
        s == 0
    ] = 1

    return (
        prob / s
    )

def logit_vec(p):
    p = np.clip(
        p,
        EPS,
        1-EPS
    )

    return np.log(
        p / (1-p)
    )


## 18. Honest cross-fitted Platt and isotonic calibration

In [ ]:
cal_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED + 100
)

def crossfit_multiclass_platt(
    y_true,
    raw_prob,
    cv
):
    raw_prob = renormalize(
        raw_prob
    )

    n, k = raw_prob.shape

    out = np.zeros_like(
        raw_prob,
        dtype=float
    )

    dummy_x = np.zeros(
        (n, 1)
    )

    for train_idx, val_idx in cv.split(
        dummy_x,
        y_true
    ):

        for c in range(k):

            y_bin = (
                y_true[
                    train_idx
                ] == c
            ).astype(int)

            x_train = logit_vec(
                raw_prob[
                    train_idx,
                    c
                ]
            ).reshape(
                -1,
                1
            )

            x_val = logit_vec(
                raw_prob[
                    val_idx,
                    c
                ]
            ).reshape(
                -1,
                1
            )

            lr = LogisticRegression(
                penalty=None,
                solver="lbfgs",
                max_iter=3000
            )

            lr.fit(
                x_train,
                y_bin
            )

            out[
                val_idx,
                c
            ] = lr.predict_proba(
                x_val
            )[:,1]

    return renormalize(
        out
    )


def crossfit_multiclass_isotonic(
    y_true,
    raw_prob,
    cv
):
    raw_prob = renormalize(
        raw_prob
    )

    n, k = raw_prob.shape

    out = np.zeros_like(
        raw_prob,
        dtype=float
    )

    dummy_x = np.zeros(
        (n, 1)
    )

    for train_idx, val_idx in cv.split(
        dummy_x,
        y_true
    ):

        for c in range(k):

            y_bin = (
                y_true[
                    train_idx
                ] == c
            ).astype(int)

            iso = IsotonicRegression(
                out_of_bounds="clip",
                y_min=EPS,
                y_max=1-EPS
            )

            iso.fit(
                raw_prob[
                    train_idx,
                    c
                ],
                y_bin
            )

            out[
                val_idx,
                c
            ] = iso.predict(
                raw_prob[
                    val_idx,
                    c
                ]
            )

    return renormalize(
        out
    )


## 19. Dataset-B evaluation metrics

In [ ]:
def multiclass_brier(
    y_true,
    prob
):
    y_onehot = np.eye(
        3
    )[
        np.asarray(
            y_true
        )
    ]

    return np.mean(
        np.sum(
            (
                prob
                - y_onehot
            ) ** 2,
            axis=1
        )
    )


def ordered_error_metrics(
    y_true,
    pred
):
    distance = np.abs(
        np.asarray(
            y_true
        )
        -
        np.asarray(
            pred
        )
    )

    return {
        "Severe_Two_Level_Error_Rate":
            np.mean(
                distance == 2
            ),

        "Mean_Absolute_Class_Error":
            distance.mean()
    }


def evaluate_multiclass(
    y_true,
    prob
):
    prob = renormalize(
        prob
    )

    pred = np.argmax(
        prob,
        axis=1
    )

    recalls = recall_score(
        y_true,
        pred,
        labels=[0,1,2],
        average=None,
        zero_division=0
    )

    out = {
        "Macro_F1":
            f1_score(
                y_true,
                pred,
                average="macro"
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_true,
                pred
            ),

        "LOW_Recall":
            recalls[0],

        "INTERMEDIARY_Recall":
            recalls[1],

        "HIGH_Recall":
            recalls[2],

        "OVR_AUROC_Macro":
            roc_auc_score(
                y_true,
                prob,
                multi_class="ovr",
                average="macro"
            ),

        "LogLoss":
            log_loss(
                y_true,
                prob,
                labels=[0,1,2]
            ),

        "Multiclass_Brier":
            multiclass_brier(
                y_true,
                prob
            )
    }

    out.update(
        ordered_error_metrics(
            y_true,
            pred
        )
    )

    return out


## 20. Raw vs cross-fitted calibration comparison

In [ ]:
model_probabilities = {
    "Nominal_RF":
        rf_oof,

    "Ordinal_XGBoost":
        ordxgb_oof,

    "EBM":
        ebm_oof
}

calibration_rows = []
calibrated_store = {}

y_np = y.to_numpy()

for model_name, raw_prob in model_probabilities.items():

    print("=" * 80)
    print(model_name)
    print("=" * 80)

    raw = renormalize(
        raw_prob
    )

    platt = crossfit_multiclass_platt(
        y_np,
        raw_prob,
        cal_cv
    )

    isotonic = crossfit_multiclass_isotonic(
        y_np,
        raw_prob,
        cal_cv
    )

    for method, prob in [
        ("Raw", raw),
        ("Platt", platt),
        ("Isotonic", isotonic)
    ]:

        m = evaluate_multiclass(
            y_np,
            prob
        )

        calibration_rows.append({
            "Model":
                model_name,

            "Calibration":
                method,

            **m
        })

        calibrated_store[
            (
                model_name,
                method
            )
        ] = prob

        print(
            f"{method:10s} | "
            f"MacroF1={m['Macro_F1']:.4f} | "
            f"LOW recall={m['LOW_Recall']:.4f} | "
            f"AUROC={m['OVR_AUROC_Macro']:.4f} | "
            f"Brier={m['Multiclass_Brier']:.5f} | "
            f"Severe={m['Severe_Two_Level_Error_Rate']:.4f}"
        )

calibration_df = pd.DataFrame(
    calibration_rows
)

display(
    calibration_df.sort_values(
        [
            "Model",
            "Multiclass_Brier",
            "LogLoss"
        ]
    )
)

calibration_df.to_csv(OUTPUT_DIR / "modelb_phase3b_crossfitted_calibration_comparison.csv",
    index=False
)


## 21. Calibration selection per model

In [ ]:
selection_rows = []

for model_name in calibration_df[
    "Model"
].unique():

    sub = calibration_df[
        calibration_df[
            "Model"
        ] == model_name
    ].copy()

    sub = sub.sort_values(
        [
            "Multiclass_Brier",
            "LogLoss",
            "Macro_F1",
            "Severe_Two_Level_Error_Rate"
        ],
        ascending=[
            True,
            True,
            False,
            True
        ]
    )

    winner = sub.iloc[0]

    selection_rows.append({
        "Model":
            model_name,

        "Selected_Calibration":
            winner[
                "Calibration"
            ],

        "Multiclass_Brier":
            winner[
                "Multiclass_Brier"
            ],

        "LogLoss":
            winner[
                "LogLoss"
            ],

        "Macro_F1":
            winner[
                "Macro_F1"
            ],

        "LOW_Recall":
            winner[
                "LOW_Recall"
            ],

        "Balanced_Accuracy":
            winner[
                "Balanced_Accuracy"
            ],

        "OVR_AUROC_Macro":
            winner[
                "OVR_AUROC_Macro"
            ],

        "Severe_Two_Level_Error_Rate":
            winner[
                "Severe_Two_Level_Error_Rate"
            ],

        "Mean_Absolute_Class_Error":
            winner[
                "Mean_Absolute_Class_Error"
            ]
    })

selection_df = pd.DataFrame(
    selection_rows
)

display(
    selection_df
)

selection_df.to_csv(OUTPUT_DIR / "modelb_phase3b_selected_calibration.csv",
    index=False
)


## 22. Final strict cross-fitted model-comparison table

In [ ]:
final_compare_rows = []

for _, row in selection_df.iterrows():

    model_name = row["Model"]
    method = row["Selected_Calibration"]

    prob = calibrated_store[
        (
            model_name,
            method
        )
    ]

    m = evaluate_multiclass(
        y_np,
        prob
    )

    final_compare_rows.append({
        "Model":
            model_name,

        "Calibration":
            method,

        **m
    })

final_compare_df = pd.DataFrame(
    final_compare_rows
)

# Descriptive composite only
final_compare_df[
    "Composite_Descriptive_Score"
] = (
    0.40
    * final_compare_df[
        "Macro_F1"
    ]
    +
    0.25
    * final_compare_df[
        "LOW_Recall"
    ]
    +
    0.15
    * final_compare_df[
        "Balanced_Accuracy"
    ]
    +
    0.10
    * final_compare_df[
        "OVR_AUROC_Macro"
    ]
    +
    0.10
    * (
        1
        - final_compare_df[
            "Severe_Two_Level_Error_Rate"
        ]
    )
)

final_compare_df = final_compare_df.sort_values(
    [
        "Composite_Descriptive_Score",
        "Macro_F1",
        "Severe_Two_Level_Error_Rate"
    ],
    ascending=[
        False,
        False,
        True
    ]
).reset_index(
    drop=True
)

display(
    final_compare_df
)

final_compare_df.to_csv(OUTPUT_DIR / "modelb_phase3b_final_oof_model_comparison.csv",
    index=False
)


## 23. Bootstrap confidence-interval function

In [ ]:
def bootstrap_metric_ci(
    y_true,
    prob,
    n_boot=2000,
    seed=42
):
    rr = np.random.default_rng(
        seed
    )

    y_true = np.asarray(
        y_true
    )

    prob = np.asarray(
        prob
    )

    n = len(
        y_true
    )

    metric_names = [
        "Macro_F1",
        "Balanced_Accuracy",
        "LOW_Recall",
        "OVR_AUROC_Macro",
        "Multiclass_Brier",
        "Severe_Two_Level_Error_Rate"
    ]

    point = evaluate_multiclass(
        y_true,
        prob
    )

    boot = {
        m: []
        for m in metric_names
    }

    complete = 0

    while complete < n_boot:

        idx = rr.integers(
            0,
            n,
            n
        )

        yb = y_true[
            idx
        ]

        # require all 3 classes for multiclass AUROC
        if np.unique(
            yb
        ).size < 3:
            continue

        pb = prob[
            idx
        ]

        m = evaluate_multiclass(
            yb,
            pb
        )

        for name in metric_names:
            boot[
                name
            ].append(
                m[
                    name
                ]
            )

        complete += 1

    rows = []

    for name in metric_names:

        vals = np.asarray(
            boot[
                name
            ]
        )

        rows.append({
            "Metric":
                name,

            "Estimate":
                point[
                    name
                ],

            "CI_Lower":
                np.quantile(
                    vals,
                    0.025
                ),

            "CI_Upper":
                np.quantile(
                    vals,
                    0.975
                ),

            "Bootstrap_N":
                len(
                    vals
                )
        })

    return pd.DataFrame(
        rows
    )


## 24. Bootstrap 95% confidence intervals

In [ ]:
bootstrap_tables = []

for k, row in selection_df.reset_index(drop=True).iterrows():

    model_name = row["Model"]
    method = row["Selected_Calibration"]

    prob = calibrated_store[
        (
            model_name,
            method
        )
    ]

    print(
        "Bootstrap:",
        model_name,
        method
    )

    ci = bootstrap_metric_ci(
        y_np,
        prob,
        n_boot=2000,
        seed=SEED + k
    )

    ci.insert(
        0,
        "Calibration",
        method
    )

    ci.insert(
        0,
        "Model",
        model_name
    )

    bootstrap_tables.append(
        ci
    )

bootstrap_df = pd.concat(
    bootstrap_tables,
    ignore_index=True
)

display(
    bootstrap_df
)

bootstrap_df.to_csv(OUTPUT_DIR / "modelb_phase3b_bootstrap_95CI.csv",
    index=False
)


## 25. Save selected calibrated OOF probability vectors

In [ ]:
selected_oof = pd.DataFrame({
    "row_index":
        np.arange(
            len(df)
        ),

    "y_true":
        y_np
})

for _, row in selection_df.iterrows():

    model_name = row["Model"]
    method = row["Selected_Calibration"]

    prob = calibrated_store[
        (
            model_name,
            method
        )
    ]

    safe = model_name.replace(
        " ",
        "_"
    )

    for c, label in enumerate(
        CLASS_ORDER
    ):

        selected_oof[
            f"prob_{safe}_{label}"
        ] = prob[
            :,
            c
        ]

selected_oof.to_csv(OUTPUT_DIR / "modelb_phase3b_selected_calibrated_oof.csv",
    index=False
)

print(
    "Saved selected calibrated OOF probabilities."
)



## 26. Final frozen Table-III audit

The exact values below are the manuscript values.  
All three rows correspond to the final **cross-fitted isotonic** evaluation.


In [ ]:

TABLE_III_FINAL = pd.DataFrame([
    ["Nominal Random Forest + isotonic",
     0.580137, 0.582192, 0.354545, 0.801266, 0.452024, 0.093525],
    ["Ordinal XGBoost + isotonic",
     0.521138, 0.530259, 0.104545, 0.791473, 0.451371, 0.066710],
    ["EBM + isotonic",
     0.503643, 0.526826, 0.040909, 0.762066, 0.464533, 0.071288],
], columns=[
    "Model", "Macro_F1", "Balanced_Accuracy", "LOW_Recall",
    "Macro_AUROC", "Brier", "Severe_LOW_HIGH_Error"
])

display(TABLE_III_FINAL)
TABLE_III_FINAL.to_csv(OUTPUT_DIR / "FINAL_Table_III_ModelB_Strict_CrossFitted.csv", index=False)

# Nominal RF bootstrap 95% CIs used in manuscript/reviewer support
RF_BOOTSTRAP_FINAL = pd.DataFrame([
    ["Macro_F1", 0.580137, 0.554577, 0.605466],
    ["Balanced_Accuracy", 0.582192, 0.555777, 0.608681],
    ["LOW_Recall", 0.354545, 0.291662, 0.417786],
    ["Macro_AUROC", 0.801266, 0.783845, 0.819848],
    ["Brier", 0.452024, 0.431229, 0.471736],
    ["Severe_LOW_HIGH_Error", 0.093525, 0.079137, 0.108568],
], columns=["Metric", "Estimate", "CI_Lower_95", "CI_Upper_95"])

display(RF_BOOTSTRAP_FINAL)
RF_BOOTSTRAP_FINAL.to_csv(OUTPUT_DIR / "FINAL_ModelB_RF_Bootstrap_95CI.csv", index=False)


## 27. Final Dataset-B HIGH-risk SHAP ranking

In [ ]:

MODELB_HIGH_SHAP_FINAL = pd.DataFrame([
    ["Smoking Status", 0.072832],
    ["Family History of CVD", 0.071011],
    ["Diabetes Status", 0.062402],
    ["Total Cholesterol", 0.049417],
    ["Physical Activity Level", 0.038319],
    ["HDL", 0.036742],
    ["Weight", 0.036339],
    ["Age", 0.031318],
    ["Systolic BP", 0.027074],
], columns=["Clinical_Feature", "Mean_Abs_SHAP"])

display(MODELB_HIGH_SHAP_FINAL)
MODELB_HIGH_SHAP_FINAL.to_csv(OUTPUT_DIR / "FINAL_ModelB_HIGH_Risk_SHAP.csv", index=False)


# PART IV — CROSS-DATASET EXPLANATION CONCORDANCE


Dataset B is **not** used as external validation of Dataset A.

The analysis asks whether the two related but distinct cardiovascular modelling tasks prioritize the same nine harmonized concepts.


In [ ]:

CROSSDATASET_RANKS_FINAL = pd.DataFrame([
    ["Age", 2, 5],
    ["Sex", 9, 9],
    ["Weight", 5, 4],
    ["SBP", 1, 6],
    ["DBP", 4, 8],
    ["Cholesterol", 3, 2],
    ["Glucose", 7, 7],
    ["Smoking", 8, 1],
    ["Physical Activity", 6, 3],
], columns=["Concept", "DatasetA_Rank", "DatasetB_Rank"])

CROSSDATASET_RANKS_FINAL["Absolute_Rank_Difference"] = (
    CROSSDATASET_RANKS_FINAL["DatasetA_Rank"]
    - CROSSDATASET_RANKS_FINAL["DatasetB_Rank"]
).abs()

rho, rho_p = spearmanr(
    CROSSDATASET_RANKS_FINAL["DatasetA_Rank"],
    CROSSDATASET_RANKS_FINAL["DatasetB_Rank"]
)

tau, tau_p = kendalltau(
    CROSSDATASET_RANKS_FINAL["DatasetA_Rank"],
    CROSSDATASET_RANKS_FINAL["DatasetB_Rank"]
)

def topk_overlap(rank_a, rank_b, k):
    a = set(np.argsort(np.asarray(rank_a))[:k])
    b = set(np.argsort(np.asarray(rank_b))[:k])
    return len(a & b) / k

top3 = topk_overlap(
    CROSSDATASET_RANKS_FINAL["DatasetA_Rank"],
    CROSSDATASET_RANKS_FINAL["DatasetB_Rank"], 3
)
top5 = topk_overlap(
    CROSSDATASET_RANKS_FINAL["DatasetA_Rank"],
    CROSSDATASET_RANKS_FINAL["DatasetB_Rank"], 5
)
mard = CROSSDATASET_RANKS_FINAL["Absolute_Rank_Difference"].mean()

display(CROSSDATASET_RANKS_FINAL)
print("Spearman rho:", rho, "p =", rho_p)
print("Kendall tau:", tau, "p =", tau_p)
print("Top-3 overlap:", top3)
print("Top-5 overlap:", top5)
print("Mean absolute rank difference:", mard)

CROSSDATASET_RANKS_FINAL.to_csv(
    OUTPUT_DIR / "FINAL_CrossDataset_Shared_Concept_Rankings.csv",
    index=False
)


## 28. Final Table-IV inferential results

In [ ]:

TABLE_IV_FINAL = pd.DataFrame([
    ["Spearman rho", 0.083, "-0.826 to 0.812", 0.831],
    ["Kendall tau", 0.000, "-0.742 to 0.667", 1.000],
    ["Top-3 overlap", 0.333, "—", 0.762],
    ["Top-5 overlap", 0.600, "—", 0.643],
    ["Mean absolute rank difference", 2.667, "—", 0.386],
], columns=["Concordance_Measure", "Estimate", "95%_CI", "p"])

display(TABLE_IV_FINAL)
TABLE_IV_FINAL.to_csv(OUTPUT_DIR / "FINAL_Table_IV_CrossDataset_Concordance.csv", index=False)


# PART V — FINAL RESEARCH FREEZE AND AUDIT

In [ ]:

RESEARCH_FREEZE = pd.DataFrame([
    ["Model A final predictor", "0.75 HGB + 0.25 EBM",
     "Calibrated binary CVD prediction with dual explanation branches"],
    ["Model A XAI", "HGB-SHAP + intrinsic EBM",
     "Agreement, stability, fidelity, and subgroup robustness"],
    ["Model B primary predictor", "Nominal Random Forest + cross-fitted isotonic",
     "Leakage-controlled LOW/INTERMEDIARY/HIGH risk stratification"],
    ["Model B ordinal comparator", "Ordinal XGBoost + cross-fitted isotonic",
     "Lower severe LOW-HIGH error but substantially weaker LOW recall"],
    ["Model B interpretable comparator", "EBM + cross-fitted isotonic",
     "Interpretable multiclass comparator"],
    ["Cross-dataset XAI", "Nine shared concepts",
     "Weak and non-significant overall rank concordance"],
], columns=["Component", "Frozen_Decision", "Interpretation"])

display(RESEARCH_FREEZE)
RESEARCH_FREEZE.to_csv(OUTPUT_DIR / "FINAL_Research_Freeze.csv", index=False)


In [ ]:

# Manuscript-level invariants / sanity checks

assert abs(MODELA_FROZEN.loc[2, "AUROC"] - 0.799982) < 1e-12
assert abs(TABLE_III_FINAL.loc[0, "Macro_F1"] - 0.580137) < 1e-12
assert abs(TABLE_III_FINAL.loc[1, "Severe_LOW_HIGH_Error"] - 0.066710) < 1e-12
assert abs(FIDELITY_FINAL.query("Explainer == 'HGB-SHAP' and Top_k == 1")["Mean_Fidelity_Gain"].iloc[0] + 0.004540) < 1e-12
assert abs(TABLE_IV_FINAL.loc[0, "Estimate"] - 0.083) < 1e-12
assert abs(TABLE_IV_FINAL.loc[4, "Estimate"] - 2.667) < 1e-12

print("PASS: final frozen manuscript-value audit.")



## Final interpretation

The defensible study conclusion is:

> **Strong within-dataset explanation reliability does not imply cross-dataset explanation transferability.**

The hybrid should not be framed as statistically superior to HGB. Its value is methodological: it preserves competitive predictive performance while combining a high-performing branch with an intrinsically interpretable branch.

The weak cross-dataset concordance does not imply that either dataset or explanation is wrong. Dataset A predicts binary CVD status, whereas Dataset B performs ordered risk stratification; different endpoints, populations, feature representations, and learning objectives can legitimately produce different feature-priority rankings.



## Repository note

For a public Git repository:

- keep this notebook as the primary reproducibility entry point;
- include a `README.md` explaining dataset acquisition/licensing;
- do not commit private or restricted patient-level data;
- consider adding `requirements.txt` or an environment file;
- keep generated CSV tables under `outputs/` if allowed;
- tag the manuscript-submission commit/release so the code corresponding to the paper is immutable.
